# DIMENSIONS

### DIMENSION AGENTE

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_dim_agente AS

WITH agentes_base AS (
    SELECT DISTINCT
        CAST(fecha AS DATE) AS fecha,
        UPPER(TRIM(codigo_agente)) AS codigo_agente,
        TRIM(nombre_agente) AS nombre_agente,

        UPPER(
            REGEXP_REPLACE(
                TRIM(nombre_agente),
                '[^A-Za-z0-9ÁÉÍÓÚáéíóúÑñ]',
                ''
            )
        ) AS nombre_agente_normalizado,

        TRIM(actividad_agente) AS actividad_agente,

        UPPER(
            REGEXP_REPLACE(
                TRIM(actividad_agente),
                '\\s+',
                ' '
            )
        ) AS actividad_normalizada

    FROM observatorio_dev.silver.agentes

    WHERE codigo_agente IS NOT NULL
      AND TRIM(codigo_agente) <> ''
      AND nombre_agente IS NOT NULL
      AND TRIM(nombre_agente) <> ''
      AND actividad_agente IS NOT NULL
      AND TRIM(actividad_agente) <> ''
),

comparacion AS (
    SELECT
        *,

        LAG(nombre_agente_normalizado) OVER (
            PARTITION BY codigo_agente
            ORDER BY fecha
        ) AS nombre_anterior_normalizado,

        LAG(actividad_normalizada) OVER (
            PARTITION BY codigo_agente
            ORDER BY fecha
        ) AS actividad_anterior_normalizada

    FROM agentes_base
),

marcacion AS (
    SELECT
        *,

        CASE
            WHEN nombre_anterior_normalizado IS NULL
                THEN 1

            WHEN nombre_agente_normalizado <> nombre_anterior_normalizado
              OR actividad_normalizada <> actividad_anterior_normalizada
                THEN 1

            ELSE 0
        END AS inicia_nueva_version

    FROM comparacion
),

versiones AS (
    SELECT
        *,

        SUM(inicia_nueva_version) OVER (
            PARTITION BY codigo_agente
            ORDER BY fecha
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS numero_version

    FROM marcacion
),

atributos_version AS (
    SELECT
        codigo_agente,
        numero_version,
        nombre_agente,
        nombre_agente_normalizado,
        actividad_agente,
        actividad_normalizada,
        fecha,

        ROW_NUMBER() OVER (
            PARTITION BY codigo_agente, numero_version
            ORDER BY fecha DESC
        ) AS posicion

    FROM versiones
),

periodos AS (
    SELECT
        codigo_agente,
        numero_version,
        MIN(fecha) AS fecha_inicio

    FROM versiones

    GROUP BY
        codigo_agente,
        numero_version
),

periodos_completos AS (
    SELECT
        p.codigo_agente,
        a.nombre_agente,
        a.nombre_agente_normalizado,
        a.actividad_agente,
        a.actividad_normalizada,
        CAST(p.numero_version AS INT) AS numero_version,
        p.fecha_inicio,

        LEAD(p.fecha_inicio) OVER (
            PARTITION BY p.codigo_agente
            ORDER BY p.numero_version
        ) AS siguiente_fecha_inicio

    FROM periodos AS p

    INNER JOIN atributos_version AS a
        ON  p.codigo_agente = a.codigo_agente
        AND p.numero_version = a.numero_version
        AND a.posicion = 1
)

SELECT
    codigo_agente,
    nombre_agente,
    nombre_agente_normalizado,
    actividad_agente,
    actividad_normalizada,
    numero_version,
    fecha_inicio,

    COALESCE(
        DATE_SUB(siguiente_fecha_inicio, 1),
        DATE '9999-12-31'
    ) AS fecha_fin,

    siguiente_fecha_inicio IS NULL AS es_actual

FROM periodos_completos;

In [0]:
%sql
MERGE INTO observatorio_dev.gold.dim_agente AS tgt

USING (
    SELECT
        codigo_agente,
        nombre_agente,
        nombre_agente_normalizado,
        actividad_agente,
        actividad_normalizada,
        numero_version,
        fecha_inicio,
        fecha_fin,
        es_actual,
        CURRENT_TIMESTAMP() AS fecha_proceso

    FROM stg_dim_agente
) AS src

ON  tgt.codigo_agente = src.codigo_agente
AND tgt.fecha_inicio = src.fecha_inicio

WHEN MATCHED AND (
       tgt.nombre_agente <> src.nombre_agente
    OR tgt.nombre_agente_normalizado <> src.nombre_agente_normalizado
    OR tgt.actividad_agente <> src.actividad_agente
    OR tgt.actividad_normalizada <> src.actividad_normalizada
    OR tgt.numero_version <> src.numero_version
    OR tgt.fecha_fin <> src.fecha_fin
    OR tgt.es_actual <> src.es_actual
)
THEN UPDATE SET
    tgt.nombre_agente = src.nombre_agente,
    tgt.nombre_agente_normalizado = src.nombre_agente_normalizado,
    tgt.actividad_agente = src.actividad_agente,
    tgt.actividad_normalizada = src.actividad_normalizada,
    tgt.numero_version = src.numero_version,
    tgt.fecha_fin = src.fecha_fin,
    tgt.es_actual = src.es_actual,
    tgt.fecha_actualizacion = src.fecha_proceso

WHEN NOT MATCHED THEN
    INSERT (
        codigo_agente,
        nombre_agente,
        nombre_agente_normalizado,
        actividad_agente,
        actividad_normalizada,
        numero_version,
        fecha_inicio,
        fecha_fin,
        es_actual,
        fecha_creacion,
        fecha_actualizacion
    )
    VALUES (
        src.codigo_agente,
        src.nombre_agente,
        src.nombre_agente_normalizado,
        src.actividad_agente,
        src.actividad_normalizada,
        src.numero_version,
        src.fecha_inicio,
        src.fecha_fin,
        src.es_actual,
        src.fecha_proceso,
        src.fecha_proceso
    );

In [0]:
%sql
SELECT
    COUNT(*) AS versiones_totales,
    COUNT(DISTINCT codigo_agente) AS agentes_distintos,
    SUM(CASE WHEN es_actual THEN 1 ELSE 0 END)
        AS versiones_actuales
FROM observatorio_dev.gold.dim_agente;

In [0]:
%sql
SELECT COUNT(*) AS versiones_faltantes
FROM stg_dim_agente AS s
LEFT JOIN observatorio_dev.gold.dim_agente AS d
    ON s.codigo_agente = d.codigo_agente
   AND s.fecha_inicio = d.fecha_inicio
WHERE d.agente_key IS NULL;

### Dimension Fecha

In [0]:
%sql
MERGE INTO observatorio_dev.gold.dim_fecha AS tgt

USING (
    WITH rango_fuente AS (
        SELECT
            COALESCE(
                MIN(CAST(fecha AS DATE)),
                CURRENT_DATE()
            ) AS fecha_minima,

            COALESCE(
                MAX(CAST(fecha AS DATE)),
                CURRENT_DATE()
            ) AS fecha_maxima

        FROM observatorio_dev.silver.agentes
    ),

    rango_calendario AS (
        SELECT
            fecha_minima AS fecha_inicio,

            GREATEST(
                fecha_maxima,
                ADD_MONTHS(CURRENT_DATE(), 24)
            ) AS fecha_fin

        FROM rango_fuente
    ),

    fechas AS (
        SELECT
            EXPLODE(
                SEQUENCE(
                    fecha_inicio,
                    fecha_fin,
                    INTERVAL 1 DAY
                )
            ) AS fecha

        FROM rango_calendario
    )

    SELECT
        CAST(
            DATE_FORMAT(fecha, 'yyyyMMdd')
            AS INT
        ) AS fecha_key,

        fecha,

        CAST(YEAR(fecha) AS SMALLINT) AS anio,

        CAST(
            CASE
                WHEN MONTH(fecha) <= 6 THEN 1
                ELSE 2
            END
            AS TINYINT
        ) AS semestre,

        CAST(QUARTER(fecha) AS TINYINT) AS trimestre,

        CAST(MONTH(fecha) AS TINYINT) AS mes_numero,

        CASE MONTH(fecha)
            WHEN 1 THEN 'Enero'
            WHEN 2 THEN 'Febrero'
            WHEN 3 THEN 'Marzo'
            WHEN 4 THEN 'Abril'
            WHEN 5 THEN 'Mayo'
            WHEN 6 THEN 'Junio'
            WHEN 7 THEN 'Julio'
            WHEN 8 THEN 'Agosto'
            WHEN 9 THEN 'Septiembre'
            WHEN 10 THEN 'Octubre'
            WHEN 11 THEN 'Noviembre'
            WHEN 12 THEN 'Diciembre'
        END AS mes_nombre,

        CASE MONTH(fecha)
            WHEN 1 THEN 'Ene'
            WHEN 2 THEN 'Feb'
            WHEN 3 THEN 'Mar'
            WHEN 4 THEN 'Abr'
            WHEN 5 THEN 'May'
            WHEN 6 THEN 'Jun'
            WHEN 7 THEN 'Jul'
            WHEN 8 THEN 'Ago'
            WHEN 9 THEN 'Sep'
            WHEN 10 THEN 'Oct'
            WHEN 11 THEN 'Nov'
            WHEN 12 THEN 'Dic'
        END AS mes_nombre_corto,

        CAST(
            YEAR(fecha) * 100 + MONTH(fecha)
            AS INT
        ) AS anio_mes,

        CONCAT(
            CASE MONTH(fecha)
                WHEN 1 THEN 'Enero'
                WHEN 2 THEN 'Febrero'
                WHEN 3 THEN 'Marzo'
                WHEN 4 THEN 'Abril'
                WHEN 5 THEN 'Mayo'
                WHEN 6 THEN 'Junio'
                WHEN 7 THEN 'Julio'
                WHEN 8 THEN 'Agosto'
                WHEN 9 THEN 'Septiembre'
                WHEN 10 THEN 'Octubre'
                WHEN 11 THEN 'Noviembre'
                WHEN 12 THEN 'Diciembre'
            END,
            ' ',
            CAST(YEAR(fecha) AS STRING)
        ) AS anio_mes_nombre,

        CAST(WEEKOFYEAR(fecha) AS TINYINT) AS semana_anio,

        CAST(DAYOFYEAR(fecha) AS SMALLINT) AS dia_anio,

        CAST(DAY(fecha) AS TINYINT) AS dia_mes,

        CAST(
            PMOD(DAYOFWEEK(fecha) + 5, 7) + 1
            AS TINYINT
        ) AS dia_semana_numero,

        CASE DAYOFWEEK(fecha)
            WHEN 1 THEN 'Domingo'
            WHEN 2 THEN 'Lunes'
            WHEN 3 THEN 'Martes'
            WHEN 4 THEN 'Miércoles'
            WHEN 5 THEN 'Jueves'
            WHEN 6 THEN 'Viernes'
            WHEN 7 THEN 'Sábado'
        END AS dia_semana_nombre,

        DAYOFWEEK(fecha) IN (1, 7) AS es_fin_semana,

        fecha = TRUNC(fecha, 'MONTH') AS es_inicio_mes,

        fecha = LAST_DAY(fecha) AS es_fin_mes,

        CURRENT_TIMESTAMP() AS fecha_proceso

    FROM fechas

) AS src

ON tgt.fecha_key = src.fecha_key

WHEN NOT MATCHED THEN
    INSERT (
        fecha_key,
        fecha,
        anio,
        semestre,
        trimestre,
        mes_numero,
        mes_nombre,
        mes_nombre_corto,
        anio_mes,
        anio_mes_nombre,
        semana_anio,
        dia_anio,
        dia_mes,
        dia_semana_numero,
        dia_semana_nombre,
        es_fin_semana,
        es_inicio_mes,
        es_fin_mes,
        fecha_creacion,
        fecha_actualizacion
    )
    VALUES (
        src.fecha_key,
        src.fecha,
        src.anio,
        src.semestre,
        src.trimestre,
        src.mes_numero,
        src.mes_nombre,
        src.mes_nombre_corto,
        src.anio_mes,
        src.anio_mes_nombre,
        src.semana_anio,
        src.dia_anio,
        src.dia_mes,
        src.dia_semana_numero,
        src.dia_semana_nombre,
        src.es_fin_semana,
        src.es_inicio_mes,
        src.es_fin_mes,
        src.fecha_proceso,
        src.fecha_proceso
    );

### Dimension Embalses

In [0]:
%sql
DESCRIBE TABLE observatorio_dev.silver.embalses;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_dim_embalse AS

WITH embalses_base AS (
    SELECT
        id,

        UPPER(TRIM(codigo_embalse)) AS codigo_embalse,

        TRIM(nombre_embalse) AS nombre_embalse,

        UPPER(
            REGEXP_REPLACE(
                TRIM(nombre_embalse),
                '[^A-Za-z0-9ÁÉÍÓÚáéíóúÑñ]',
                ''
            )
        ) AS nombre_embalse_normalizado,

        CAST(latitud AS DECIMAL(9,6)) AS latitud,

        CAST(longitud AS DECIMAL(10,6)) AS longitud,

        source_file_name,
        ingestion_timestamp,
        load_date,
        silver_updated_at

    FROM observatorio_dev.silver.embalses

    WHERE codigo_embalse IS NOT NULL
      AND TRIM(codigo_embalse) <> ''
      AND nombre_embalse IS NOT NULL
      AND TRIM(nombre_embalse) <> ''
),

embalses_clasificados AS (
    SELECT
        *,

        CASE
            WHEN latitud IS NOT NULL
             AND longitud IS NOT NULL
             AND latitud BETWEEN -90 AND 90
             AND longitud BETWEEN -180 AND 180
                THEN TRUE
            ELSE FALSE
        END AS coordenada_valida,

        ROW_NUMBER() OVER (
            PARTITION BY codigo_embalse
            ORDER BY
                silver_updated_at DESC NULLS LAST,
                ingestion_timestamp DESC NULLS LAST,
                load_date DESC NULLS LAST,
                id DESC
        ) AS posicion

    FROM embalses_base
)

SELECT
    codigo_embalse,
    nombre_embalse,
    nombre_embalse_normalizado,
    latitud,
    longitud,
    coordenada_valida,
    NOT coordenada_valida AS requiere_revision_manual,
    source_file_name,
    load_date AS silver_load_date,
    silver_updated_at

FROM embalses_clasificados

WHERE posicion = 1;

In [0]:
%sql
MERGE INTO observatorio_dev.gold.dim_embalse AS tgt

USING (
    SELECT
        codigo_embalse,
        nombre_embalse,
        nombre_embalse_normalizado,
        latitud,
        longitud,
        coordenada_valida,
        requiere_revision_manual,
        source_file_name,
        silver_load_date,
        silver_updated_at,
        CURRENT_TIMESTAMP() AS fecha_proceso

    FROM stg_dim_embalse
) AS src

ON tgt.codigo_embalse = src.codigo_embalse

WHEN MATCHED AND (
       NOT (tgt.nombre_embalse <=> src.nombre_embalse)
    OR NOT (
        tgt.nombre_embalse_normalizado
        <=>
        src.nombre_embalse_normalizado
    )
    OR NOT (tgt.latitud <=> src.latitud)
    OR NOT (tgt.longitud <=> src.longitud)
    OR NOT (tgt.coordenada_valida <=> src.coordenada_valida)
    OR NOT (
        tgt.requiere_revision_manual
        <=>
        src.requiere_revision_manual
    )
    OR NOT (tgt.source_file_name <=> src.source_file_name)
    OR NOT (tgt.silver_load_date <=> src.silver_load_date)
    OR NOT (tgt.silver_updated_at <=> src.silver_updated_at)
)
THEN UPDATE SET
    tgt.nombre_embalse = src.nombre_embalse,
    tgt.nombre_embalse_normalizado =
        src.nombre_embalse_normalizado,
    tgt.latitud = src.latitud,
    tgt.longitud = src.longitud,
    tgt.coordenada_valida = src.coordenada_valida,
    tgt.requiere_revision_manual =
        src.requiere_revision_manual,
    tgt.source_file_name = src.source_file_name,
    tgt.silver_load_date = src.silver_load_date,
    tgt.silver_updated_at = src.silver_updated_at,
    tgt.fecha_actualizacion = src.fecha_proceso

WHEN NOT MATCHED THEN
    INSERT (
        codigo_embalse,
        nombre_embalse,
        nombre_embalse_normalizado,
        latitud,
        longitud,
        coordenada_valida,
        requiere_revision_manual,
        source_file_name,
        silver_load_date,
        silver_updated_at,
        fecha_creacion,
        fecha_actualizacion
    )
    VALUES (
        src.codigo_embalse,
        src.nombre_embalse,
        src.nombre_embalse_normalizado,
        src.latitud,
        src.longitud,
        src.coordenada_valida,
        src.requiere_revision_manual,
        src.source_file_name,
        src.silver_load_date,
        src.silver_updated_at,
        src.fecha_proceso,
        src.fecha_proceso
    );

## DISEÑOS

# FACTS

In [0]:
%sql
SELECT
    codigo_planta,
    COUNT(DISTINCT codigo_sic_agente) AS agentes_distintos,
    COLLECT_SET(codigo_sic_agente) AS agentes
FROM observatorio_dev.silver.plantas
GROUP BY codigo_planta
HAVING COUNT(DISTINCT codigo_sic_agente) > 1
ORDER BY agentes_distintos DESC;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_dim_planta_maestro AS

WITH plantas_base AS (
    SELECT
        id,
        CAST(fecha AS DATE) AS fecha,

        UPPER(TRIM(codigo_planta)) AS codigo_planta,

        TRIM(nombre_planta) AS nombre_planta,

        UPPER(
            REGEXP_REPLACE(
                TRIM(nombre_planta),
                '[^A-Za-z0-9ÁÉÍÓÚáéíóúÑñ]',
                ''
            )
        ) AS nombre_planta_comparacion,

        NULLIF(
            UPPER(TRIM(codigo_sic_agente)),
            ''
        ) AS codigo_sic_agente,

        CAST(
            cap_efectiva_neta AS DECIMAL(18,4)
        ) AS cap_efectiva_neta,

        CAST(fpo AS DATE) AS fpo,

        NULLIF(
            UPPER(TRIM(codigo_sub_area_operativa)),
            ''
        ) AS codigo_sub_area_operativa,

        NULLIF(
            UPPER(TRIM(codigo_area_operativa)),
            ''
        ) AS codigo_area_operativa,

        NULLIF(
            UPPER(
                REGEXP_REPLACE(
                    TRIM(tipo_despacho_recurso),
                    '\\s+',
                    ' '
                )
            ),
            ''
        ) AS tipo_despacho_recurso,

        NULLIF(
            UPPER(
                REGEXP_REPLACE(
                    TRIM(tipo_clasificacion),
                    '\\s+',
                    ' '
                )
            ),
            ''
        ) AS tipo_clasificacion,

        NULLIF(
            UPPER(
                REGEXP_REPLACE(
                    TRIM(tipo_generacion),
                    '\\s+',
                    ' '
                )
            ),
            ''
        ) AS tipo_generacion,

        ingestion_timestamp,
        load_date,
        silver_updated_at

    FROM observatorio_dev.silver.plantas

    WHERE codigo_planta IS NOT NULL
      AND TRIM(codigo_planta) <> ''
      AND nombre_planta IS NOT NULL
      AND TRIM(nombre_planta) <> ''
      AND fecha IS NOT NULL
),

snapshots_deduplicados AS (
    SELECT
        *,

        ROW_NUMBER() OVER (
            PARTITION BY codigo_planta, fecha
            ORDER BY
                silver_updated_at DESC NULLS LAST,
                ingestion_timestamp DESC NULLS LAST,
                load_date DESC NULLS LAST,
                id DESC
        ) AS posicion_snapshot

    FROM plantas_base
),

snapshots AS (
    SELECT
        fecha,
        codigo_planta,
        nombre_planta,
        nombre_planta_comparacion,
        codigo_sic_agente,
        cap_efectiva_neta,
        fpo,
        codigo_sub_area_operativa,
        codigo_area_operativa,
        tipo_despacho_recurso,
        tipo_clasificacion,
        tipo_generacion,

        SHA2(
            CONCAT_WS(
                '||',
                COALESCE(nombre_planta_comparacion, '∅'),
                COALESCE(codigo_sic_agente, '∅'),
                COALESCE(CAST(cap_efectiva_neta AS STRING), '∅'),
                COALESCE(CAST(fpo AS STRING), '∅'),
                COALESCE(codigo_sub_area_operativa, '∅'),
                COALESCE(codigo_area_operativa, '∅'),
                COALESCE(tipo_despacho_recurso, '∅'),
                COALESCE(tipo_clasificacion, '∅'),
                COALESCE(tipo_generacion, '∅')
            ),
            256
        ) AS estado_hash

    FROM snapshots_deduplicados

    WHERE posicion_snapshot = 1
),

comparacion AS (
    SELECT
        *,

        LAG(estado_hash) OVER (
            PARTITION BY codigo_planta
            ORDER BY fecha
        ) AS estado_hash_anterior

    FROM snapshots
),

marcacion_versiones AS (
    SELECT
        *,

        CASE
            WHEN estado_hash_anterior IS NULL THEN 1
            WHEN estado_hash <> estado_hash_anterior THEN 1
            ELSE 0
        END AS inicia_nueva_version

    FROM comparacion
),

versiones AS (
    SELECT
        *,

        SUM(inicia_nueva_version) OVER (
            PARTITION BY codigo_planta
            ORDER BY fecha
            ROWS BETWEEN UNBOUNDED PRECEDING
                     AND CURRENT ROW
        ) AS numero_version

    FROM marcacion_versiones
),

atributos_version AS (
    SELECT
        codigo_planta,
        numero_version,
        nombre_planta,
        codigo_sic_agente,
        cap_efectiva_neta,
        fpo,
        codigo_sub_area_operativa,
        codigo_area_operativa,
        tipo_despacho_recurso,
        tipo_clasificacion,
        tipo_generacion,
        fecha,

        ROW_NUMBER() OVER (
            PARTITION BY codigo_planta, numero_version
            ORDER BY fecha DESC
        ) AS posicion_atributos

    FROM versiones
),

periodos AS (
    SELECT
        codigo_planta,
        numero_version,
        MIN(fecha) AS fecha_inicio_observada

    FROM versiones

    GROUP BY
        codigo_planta,
        numero_version
),

periodos_completos AS (
    SELECT
        p.codigo_planta,
        a.nombre_planta,
        a.codigo_sic_agente,
        a.cap_efectiva_neta,
        a.fpo,
        a.codigo_sub_area_operativa,
        a.codigo_area_operativa,
        a.tipo_despacho_recurso,
        a.tipo_clasificacion,
        a.tipo_generacion,

        CAST(p.numero_version AS INT) AS numero_version,

        p.fecha_inicio_observada,

        LEAD(p.fecha_inicio_observada) OVER (
            PARTITION BY p.codigo_planta
            ORDER BY p.numero_version
        ) AS siguiente_fecha_inicio

    FROM periodos AS p

    INNER JOIN atributos_version AS a
        ON p.codigo_planta = a.codigo_planta
       AND p.numero_version = a.numero_version
       AND a.posicion_atributos = 1
)

SELECT
    codigo_planta,
    nombre_planta,
    codigo_sic_agente,
    cap_efectiva_neta,
    fpo,
    codigo_sub_area_operativa,
    codigo_area_operativa,
    tipo_despacho_recurso,
    tipo_clasificacion,
    tipo_generacion,

    FALSE AS es_registro_inferido,

    'MAESTRO_PLANTAS' AS origen_registro,

    numero_version,

    CASE
        WHEN numero_version = 1
            THEN DATE '2026-01-01'
        ELSE fecha_inicio_observada
    END AS fecha_inicio,

    COALESCE(
        DATE_SUB(siguiente_fecha_inicio, 1),
        DATE '9999-12-31'
    ) AS fecha_fin,

    siguiente_fecha_inicio IS NULL AS es_actual

FROM periodos_completos;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_dim_planta_inferida AS

WITH recursos_operativos AS (

    /* Recursos observados en generación real. */
    SELECT
        UPPER(TRIM(codigo_planta)) AS codigo_planta,

        CASE
            WHEN codigo_agente IS NOT NULL
             AND TRIM(codigo_agente) <> ''
            THEN UPPER(TRIM(codigo_agente))
        END AS codigo_agente,

        CAST(fecha_hora AS DATE) AS fecha_observada,

        TRUE  AS aparece_generacion,
        FALSE AS aparece_disponibilidad,
        FALSE AS aparece_nivel_embalse

    FROM observatorio_dev.silver.generacion_real

    WHERE UPPER(TRIM(codigo_variable)) = 'GREAL'
      AND UPPER(TRIM(codigo_duracion)) = 'PT1H'
      AND UPPER(TRIM(unidad_medida)) = 'KWH'
      AND codigo_planta IS NOT NULL
      AND TRIM(codigo_planta) <> ''

    UNION ALL

    /* Recursos observados en disponibilidad. */
    SELECT
        UPPER(TRIM(codigo_planta)) AS codigo_planta,

        CAST(NULL AS STRING) AS codigo_agente,

        CAST(fecha_hora AS DATE) AS fecha_observada,

        FALSE AS aparece_generacion,
        TRUE  AS aparece_disponibilidad,
        FALSE AS aparece_nivel_embalse

    FROM observatorio_dev.silver.disponibilidad_plantas

    WHERE UPPER(TRIM(codigo_variable)) = 'DISPREAL'
      AND UPPER(TRIM(codigo_duracion)) = 'PT1H'
      AND UPPER(TRIM(unidad_medida)) = 'KWH'
      AND codigo_planta IS NOT NULL
      AND TRIM(codigo_planta) <> ''

    UNION ALL

    /* Recursos observados en nivel de energía embalsada. */
    SELECT
        UPPER(TRIM(codigo_planta)) AS codigo_planta,

        CAST(NULL AS STRING) AS codigo_agente,

        CAST(fecha_inicio AS DATE) AS fecha_observada,

        FALSE AS aparece_generacion,
        FALSE AS aparece_disponibilidad,
        TRUE  AS aparece_nivel_embalse

    FROM observatorio_dev.silver.niveles_embalses

    WHERE UPPER(TRIM(codigo_variable)) = 'NEM'
      AND UPPER(TRIM(codigo_duracion)) = 'P1D'
      AND UPPER(TRIM(unidad_medida)) = 'KWH'
      AND codigo_planta IS NOT NULL
      AND TRIM(codigo_planta) <> ''
),

recursos_consolidados AS (
    SELECT
        codigo_planta,

        COUNT(DISTINCT codigo_agente)
            AS agentes_distintos,

        CASE
            WHEN COUNT(DISTINCT codigo_agente) = 1
                THEN MAX(codigo_agente)
            ELSE NULL
        END AS codigo_sic_agente,

        MIN(fecha_observada)
            AS primera_fecha_observada,

        MAX(fecha_observada)
            AS ultima_fecha_observada,

        MAX(
            CASE WHEN aparece_generacion THEN 1 ELSE 0 END
        ) AS tiene_generacion,

        MAX(
            CASE WHEN aparece_disponibilidad THEN 1 ELSE 0 END
        ) AS tiene_disponibilidad,

        MAX(
            CASE WHEN aparece_nivel_embalse THEN 1 ELSE 0 END
        ) AS tiene_nivel_embalse

    FROM recursos_operativos

    GROUP BY codigo_planta
),

codigos_maestro AS (
    SELECT DISTINCT codigo_planta
    FROM stg_dim_planta_maestro
),

recursos_faltantes AS (
    SELECT
        r.*

    FROM recursos_consolidados AS r

    LEFT JOIN codigos_maestro AS m
        ON r.codigo_planta = m.codigo_planta

    WHERE m.codigo_planta IS NULL
)

SELECT
    codigo_planta,

    CONCAT(
        'RECURSO SIN MAESTRO - ',
        codigo_planta
    ) AS nombre_planta,

    codigo_sic_agente,

    CAST(NULL AS DECIMAL(18,4))
        AS cap_efectiva_neta,

    CAST(NULL AS DATE)
        AS fpo,

    CAST(NULL AS STRING)
        AS codigo_sub_area_operativa,

    CAST(NULL AS STRING)
        AS codigo_area_operativa,

    CAST(NULL AS STRING)
        AS tipo_despacho_recurso,

    CAST(NULL AS STRING)
        AS tipo_clasificacion,

    CAST(NULL AS STRING)
        AS tipo_generacion,

    TRUE AS es_registro_inferido,

    CONCAT_WS(
        '_Y_',

        CASE
            WHEN tiene_generacion = 1
            THEN 'GENERACION_REAL'
        END,

        CASE
            WHEN tiene_disponibilidad = 1
            THEN 'DISPONIBILIDAD'
        END,

        CASE
            WHEN tiene_nivel_embalse = 1
            THEN 'NIVEL_EMBALSE'
        END
    ) AS origen_registro,

    1 AS numero_version,

    DATE '2026-01-01'
        AS fecha_inicio,

    DATE '9999-12-31'
        AS fecha_fin,

    TRUE AS es_actual

FROM recursos_faltantes;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_dim_planta AS

SELECT
    codigo_planta,
    nombre_planta,
    codigo_sic_agente,
    cap_efectiva_neta,
    fpo,
    codigo_sub_area_operativa,
    codigo_area_operativa,
    tipo_despacho_recurso,
    tipo_clasificacion,
    tipo_generacion,
    es_registro_inferido,
    origen_registro,
    numero_version,
    fecha_inicio,
    fecha_fin,
    es_actual

FROM stg_dim_planta_maestro

UNION ALL

SELECT
    codigo_planta,
    nombre_planta,
    codigo_sic_agente,
    cap_efectiva_neta,
    fpo,
    codigo_sub_area_operativa,
    codigo_area_operativa,
    tipo_despacho_recurso,
    tipo_clasificacion,
    tipo_generacion,
    es_registro_inferido,
    origen_registro,
    numero_version,
    fecha_inicio,
    fecha_fin,
    es_actual

FROM stg_dim_planta_inferida;

In [0]:
%sql
MERGE INTO observatorio_dev.gold.dim_planta AS tgt

USING (
    SELECT
        *,
        CURRENT_TIMESTAMP() AS fecha_proceso
    FROM stg_dim_planta
) AS src

ON  tgt.codigo_planta = src.codigo_planta
AND tgt.fecha_inicio = src.fecha_inicio

WHEN MATCHED AND (
       NOT (tgt.nombre_planta <=> src.nombre_planta)
    OR NOT (tgt.codigo_sic_agente <=> src.codigo_sic_agente)
    OR NOT (tgt.cap_efectiva_neta <=> src.cap_efectiva_neta)
    OR NOT (tgt.fpo <=> src.fpo)
    OR NOT (
        tgt.codigo_sub_area_operativa
        <=>
        src.codigo_sub_area_operativa
    )
    OR NOT (
        tgt.codigo_area_operativa
        <=>
        src.codigo_area_operativa
    )
    OR NOT (
        tgt.tipo_despacho_recurso
        <=>
        src.tipo_despacho_recurso
    )
    OR NOT (
        tgt.tipo_clasificacion
        <=>
        src.tipo_clasificacion
    )
    OR NOT (
        tgt.tipo_generacion
        <=>
        src.tipo_generacion
    )
    OR NOT (
        tgt.es_registro_inferido
        <=>
        src.es_registro_inferido
    )
    OR NOT (
        tgt.origen_registro
        <=>
        src.origen_registro
    )
    OR tgt.numero_version <> src.numero_version
    OR tgt.fecha_fin <> src.fecha_fin
    OR tgt.es_actual <> src.es_actual
)
THEN UPDATE SET
    tgt.nombre_planta =
        src.nombre_planta,

    tgt.codigo_sic_agente =
        src.codigo_sic_agente,

    tgt.cap_efectiva_neta =
        src.cap_efectiva_neta,

    tgt.fpo =
        src.fpo,

    tgt.codigo_sub_area_operativa =
        src.codigo_sub_area_operativa,

    tgt.codigo_area_operativa =
        src.codigo_area_operativa,

    tgt.tipo_despacho_recurso =
        src.tipo_despacho_recurso,

    tgt.tipo_clasificacion =
        src.tipo_clasificacion,

    tgt.tipo_generacion =
        src.tipo_generacion,

    tgt.es_registro_inferido =
        src.es_registro_inferido,

    tgt.origen_registro =
        src.origen_registro,

    tgt.numero_version =
        src.numero_version,

    tgt.fecha_fin =
        src.fecha_fin,

    tgt.es_actual =
        src.es_actual,

    tgt.fecha_actualizacion =
        src.fecha_proceso

WHEN NOT MATCHED THEN
INSERT (
    codigo_planta,
    nombre_planta,
    codigo_sic_agente,
    cap_efectiva_neta,
    fpo,
    codigo_sub_area_operativa,
    codigo_area_operativa,
    tipo_despacho_recurso,
    tipo_clasificacion,
    tipo_generacion,
    es_registro_inferido,
    origen_registro,
    numero_version,
    fecha_inicio,
    fecha_fin,
    es_actual,
    fecha_creacion,
    fecha_actualizacion
)
VALUES (
    src.codigo_planta,
    src.nombre_planta,
    src.codigo_sic_agente,
    src.cap_efectiva_neta,
    src.fpo,
    src.codigo_sub_area_operativa,
    src.codigo_area_operativa,
    src.tipo_despacho_recurso,
    src.tipo_clasificacion,
    src.tipo_generacion,
    src.es_registro_inferido,
    src.origen_registro,
    src.numero_version,
    src.fecha_inicio,
    src.fecha_fin,
    src.es_actual,
    src.fecha_proceso,
    src.fecha_proceso
);

In [0]:
%sql
MERGE INTO observatorio_dev.gold.dim_periodo AS tgt

USING (
    WITH horas AS (
        SELECT
            EXPLODE(SEQUENCE(0, 23)) AS hora_inicio
    )

    SELECT
        CAST(hora_inicio + 1 AS TINYINT) AS periodo_key,

        CAST(hora_inicio + 1 AS TINYINT) AS numero_periodo,

        CAST(hora_inicio AS TINYINT) AS hora_inicio,

        CAST(hora_inicio + 1 AS TINYINT) AS hora_fin,

        FORMAT_STRING(
            '%02d:00',
            hora_inicio
        ) AS hora_inicio_etiqueta,

        FORMAT_STRING(
            '%02d:00',
            hora_inicio + 1
        ) AS hora_fin_etiqueta,

        CONCAT(
            'P',
            CAST(hora_inicio + 1 AS STRING)
        ) AS periodo_etiqueta,

        CONCAT(
            FORMAT_STRING('%02d:00', hora_inicio),
            ' - ',
            FORMAT_STRING('%02d:00', hora_inicio + 1)
        ) AS rango_horario,

        CURRENT_TIMESTAMP() AS fecha_proceso

    FROM horas
) AS src

ON tgt.periodo_key = src.periodo_key

WHEN MATCHED AND (
       tgt.numero_periodo <> src.numero_periodo
    OR tgt.hora_inicio <> src.hora_inicio
    OR tgt.hora_fin <> src.hora_fin
    OR tgt.hora_inicio_etiqueta <> src.hora_inicio_etiqueta
    OR tgt.hora_fin_etiqueta <> src.hora_fin_etiqueta
    OR tgt.periodo_etiqueta <> src.periodo_etiqueta
    OR tgt.rango_horario <> src.rango_horario
)
THEN UPDATE SET
    tgt.numero_periodo = src.numero_periodo,
    tgt.hora_inicio = src.hora_inicio,
    tgt.hora_fin = src.hora_fin,
    tgt.hora_inicio_etiqueta = src.hora_inicio_etiqueta,
    tgt.hora_fin_etiqueta = src.hora_fin_etiqueta,
    tgt.periodo_etiqueta = src.periodo_etiqueta,
    tgt.rango_horario = src.rango_horario

WHEN NOT MATCHED THEN
    INSERT (
        periodo_key,
        numero_periodo,
        hora_inicio,
        hora_fin,
        hora_inicio_etiqueta,
        hora_fin_etiqueta,
        periodo_etiqueta,
        rango_horario,
        fecha_creacion
    )
    VALUES (
        src.periodo_key,
        src.numero_periodo,
        src.hora_inicio,
        src.hora_fin,
        src.hora_inicio_etiqueta,
        src.hora_fin_etiqueta,
        src.periodo_etiqueta,
        src.rango_horario,
        src.fecha_proceso
    );

# Hechos

In [0]:
%sql
SET TIME ZONE 'UTC';

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_fact_generacion_real AS

WITH generacion_base AS (
    SELECT
        id,
        fecha_hora,

        UPPER(TRIM(codigo_planta))
            AS codigo_planta,

        UPPER(TRIM(codigo_agente))
            AS codigo_agente,

        UPPER(TRIM(codigo_variable))
            AS codigo_variable,

        UPPER(TRIM(version))
            AS version,

        CAST(valor AS DECIMAL(20,4))
            AS generacion_real_kwh,

        ingestion_timestamp,
        load_date,
        silver_updated_at,

        CASE
            WHEN UPPER(TRIM(version)) RLIKE '^TX[0-9]+$'
             AND CAST(
                    REGEXP_EXTRACT(
                        UPPER(TRIM(version)),
                        '^TX([0-9]+)$',
                        1
                    ) AS INT
                 ) >= 3
            THEN 1000 + CAST(
                REGEXP_EXTRACT(
                    UPPER(TRIM(version)),
                    '^TX([0-9]+)$',
                    1
                ) AS INT
            )

            WHEN UPPER(TRIM(version)) = 'TXF' THEN 900
            WHEN UPPER(TRIM(version)) = 'TXR' THEN 800
            WHEN UPPER(TRIM(version)) = 'TX2' THEN 200
            WHEN UPPER(TRIM(version)) = 'TX1' THEN 100
            ELSE 0
        END AS prioridad_version

    FROM observatorio_dev.silver.generacion_real

    WHERE UPPER(TRIM(codigo_variable)) = 'GREAL'
      AND UPPER(TRIM(codigo_duracion)) = 'PT1H'
      AND UPPER(TRIM(unidad_medida)) = 'KWH'
      AND fecha_hora IS NOT NULL
      AND codigo_planta IS NOT NULL
      AND TRIM(codigo_planta) <> ''
      AND codigo_agente IS NOT NULL
      AND TRIM(codigo_agente) <> ''
      AND valor IS NOT NULL
),

generacion_versionada AS (
    SELECT
        *,

        ROW_NUMBER() OVER (
            PARTITION BY
                fecha_hora,
                codigo_planta,
                codigo_agente,
                codigo_variable

            ORDER BY
                prioridad_version DESC,
                silver_updated_at DESC NULLS LAST,
                ingestion_timestamp DESC NULLS LAST,
                load_date DESC NULLS LAST,
                id DESC
        ) AS posicion_version

    FROM generacion_base
),

mediciones AS (
    SELECT
        SHA2(
            CONCAT_WS(
                '||',
                DATE_FORMAT(
                    fecha_hora,
                    'yyyy-MM-dd HH:mm:ss'
                ),
                codigo_planta,
                codigo_agente,
                codigo_variable
            ),
            256
        ) AS generacion_key,

        CAST(
            DATE_FORMAT(fecha_hora, 'yyyyMMdd')
            AS INT
        ) AS fecha_key,

        CAST(
            HOUR(fecha_hora) + 1
            AS TINYINT
        ) AS periodo_key,

        fecha_hora,
        codigo_planta,
        codigo_agente,
        generacion_real_kwh,
        version AS version_seleccionada,
        prioridad_version

    FROM generacion_versionada

    WHERE posicion_version = 1
)

SELECT
    m.generacion_key,
    m.fecha_key,
    m.periodo_key,
    p.planta_key,
    a.agente_key,
    m.fecha_hora,
    m.generacion_real_kwh,
    m.version_seleccionada,
    m.prioridad_version

FROM mediciones AS m

LEFT JOIN observatorio_dev.gold.dim_planta AS p
    ON m.codigo_planta = p.codigo_planta
   AND CAST(m.fecha_hora AS DATE)
       BETWEEN p.fecha_inicio AND p.fecha_fin

LEFT JOIN observatorio_dev.gold.dim_agente AS a
    ON m.codigo_agente = a.codigo_agente
   AND CAST(m.fecha_hora AS DATE)
       BETWEEN a.fecha_inicio AND a.fecha_fin;

In [0]:
%sql
SELECT
    COUNT(*) AS registros_staging,

    SUM(
        CASE WHEN planta_key IS NULL THEN 1 ELSE 0 END
    ) AS plantas_sin_dimension,

    SUM(
        CASE WHEN agente_key IS NULL THEN 1 ELSE 0 END
    ) AS agentes_sin_dimension

FROM stg_fact_generacion_real;

In [0]:
%sql
MERGE INTO observatorio_dev.gold.fact_generacion_real AS tgt

USING (
    SELECT
        generacion_key,
        fecha_key,
        periodo_key,
        planta_key,
        agente_key,
        fecha_hora,
        generacion_real_kwh,
        version_seleccionada,
        prioridad_version,
        CURRENT_TIMESTAMP() AS fecha_proceso

    FROM stg_fact_generacion_real

    WHERE planta_key IS NOT NULL
      AND agente_key IS NOT NULL
) AS src

ON tgt.generacion_key = src.generacion_key

WHEN MATCHED AND (
       NOT (tgt.fecha_key <=> src.fecha_key)
    OR NOT (tgt.periodo_key <=> src.periodo_key)
    OR NOT (tgt.planta_key <=> src.planta_key)
    OR NOT (tgt.agente_key <=> src.agente_key)
    OR NOT (
        tgt.generacion_real_kwh
        <=>
        src.generacion_real_kwh
    )
    OR NOT (
        tgt.version_seleccionada
        <=>
        src.version_seleccionada
    )
    OR NOT (
        tgt.prioridad_version
        <=>
        src.prioridad_version
    )
)
THEN UPDATE SET
    tgt.fecha_key =
        src.fecha_key,

    tgt.periodo_key =
        src.periodo_key,

    tgt.planta_key =
        src.planta_key,

    tgt.agente_key =
        src.agente_key,

    tgt.fecha_hora =
        src.fecha_hora,

    tgt.generacion_real_kwh =
        src.generacion_real_kwh,

    tgt.version_seleccionada =
        src.version_seleccionada,

    tgt.prioridad_version =
        src.prioridad_version,

    tgt.fecha_actualizacion =
        src.fecha_proceso

WHEN NOT MATCHED THEN
INSERT (
    generacion_key,
    fecha_key,
    periodo_key,
    planta_key,
    agente_key,
    fecha_hora,
    generacion_real_kwh,
    version_seleccionada,
    prioridad_version,
    fecha_creacion,
    fecha_actualizacion
)
VALUES (
    src.generacion_key,
    src.fecha_key,
    src.periodo_key,
    src.planta_key,
    src.agente_key,
    src.fecha_hora,
    src.generacion_real_kwh,
    src.version_seleccionada,
    src.prioridad_version,
    src.fecha_proceso,
    src.fecha_proceso
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW
stg_fact_disponibilidad_planta AS

WITH disponibilidad_base AS (
    SELECT
        id,

        fecha_hora,

        UPPER(TRIM(codigo_planta))
            AS codigo_planta,

        UPPER(TRIM(codigo_variable))
            AS codigo_variable,

        UPPER(TRIM(version))
            AS version,

        CAST(valor AS DECIMAL(20,4))
            AS disponibilidad_real_kwh,

        ingestion_timestamp,
        load_date,
        silver_updated_at,

        CASE
            /*
            TX3, TX4, TX5... son ajustes posteriores.
            Una versión numérica mayor tiene mayor prioridad.
            */
            WHEN UPPER(TRIM(version)) RLIKE '^TX[0-9]+$'
             AND CAST(
                    REGEXP_EXTRACT(
                        UPPER(TRIM(version)),
                        '^TX([0-9]+)$',
                        1
                    ) AS INT
                 ) >= 3
            THEN
                1000
                + CAST(
                    REGEXP_EXTRACT(
                        UPPER(TRIM(version)),
                        '^TX([0-9]+)$',
                        1
                    ) AS INT
                )

            WHEN UPPER(TRIM(version)) = 'TXF'
                THEN 900

            WHEN UPPER(TRIM(version)) = 'TXR'
                THEN 800

            WHEN UPPER(TRIM(version)) = 'TX2'
                THEN 200

            WHEN UPPER(TRIM(version)) = 'TX1'
                THEN 100

            ELSE 0
        END AS prioridad_version

    FROM observatorio_dev.silver.disponibilidad_plantas

    WHERE UPPER(TRIM(codigo_variable)) = 'DISPREAL'
      AND UPPER(TRIM(codigo_duracion)) = 'PT1H'
      AND UPPER(TRIM(unidad_medida)) = 'KWH'
      AND fecha_hora IS NOT NULL
      AND codigo_planta IS NOT NULL
      AND TRIM(codigo_planta) <> ''
      AND valor IS NOT NULL
),

/*
Selecciona una única versión por planta y hora.
*/
disponibilidad_versionada AS (
    SELECT
        *,

        ROW_NUMBER() OVER (
            PARTITION BY
                fecha_hora,
                codigo_planta,
                codigo_variable

            ORDER BY
                prioridad_version DESC,
                silver_updated_at DESC NULLS LAST,
                ingestion_timestamp DESC NULLS LAST,
                load_date DESC NULLS LAST,
                id DESC
        ) AS posicion_version

    FROM disponibilidad_base
),

mediciones AS (
    SELECT
        SHA2(
            CONCAT_WS(
                '||',
                DATE_FORMAT(
                    fecha_hora,
                    'yyyy-MM-dd HH:mm:ss'
                ),
                codigo_planta,
                codigo_variable
            ),
            256
        ) AS disponibilidad_key,

        CAST(
            DATE_FORMAT(
                fecha_hora,
                'yyyyMMdd'
            ) AS INT
        ) AS fecha_key,

        CAST(
            HOUR(fecha_hora) + 1
            AS TINYINT
        ) AS periodo_key,

        fecha_hora,
        codigo_planta,
        disponibilidad_real_kwh,

        version
            AS version_seleccionada,

        prioridad_version

    FROM disponibilidad_versionada

    WHERE posicion_version = 1
)

SELECT
    m.disponibilidad_key,
    m.fecha_key,
    m.periodo_key,
    p.planta_key,
    m.fecha_hora,
    m.disponibilidad_real_kwh,
    m.version_seleccionada,
    m.prioridad_version

FROM mediciones AS m

LEFT JOIN observatorio_dev.gold.dim_planta AS p
    ON m.codigo_planta = p.codigo_planta

   AND CAST(m.fecha_hora AS DATE)
       BETWEEN p.fecha_inicio
           AND p.fecha_fin;

In [0]:
%sql
SELECT
    COUNT(*) AS registros_staging,

    COUNT(DISTINCT disponibilidad_key)
        AS claves_distintas,

    SUM(
        CASE
            WHEN planta_key IS NULL
            THEN 1
            ELSE 0
        END
    ) AS plantas_sin_dimension,

    SUM(
        CASE
            WHEN fecha_key IS NULL
            THEN 1
            ELSE 0
        END
    ) AS fechas_nulas,

    SUM(
        CASE
            WHEN periodo_key IS NULL
            THEN 1
            ELSE 0
        END
    ) AS periodos_nulos

FROM stg_fact_disponibilidad_planta;

In [0]:
%sql
SELECT
    fecha_hora,
    planta_key,
    COUNT(*) AS cantidad

FROM stg_fact_disponibilidad_planta

GROUP BY
    fecha_hora,
    planta_key

HAVING COUNT(*) > 1;

In [0]:
%sql
MERGE INTO
observatorio_dev.gold.fact_disponibilidad_planta AS tgt

USING (
    SELECT
        disponibilidad_key,
        fecha_key,
        periodo_key,
        planta_key,
        fecha_hora,
        disponibilidad_real_kwh,
        version_seleccionada,
        prioridad_version,
        CURRENT_TIMESTAMP() AS fecha_proceso

    FROM stg_fact_disponibilidad_planta

    WHERE planta_key IS NOT NULL
) AS src

ON tgt.disponibilidad_key =
   src.disponibilidad_key

WHEN MATCHED AND (
       NOT (
            tgt.fecha_key
            <=>
            src.fecha_key
       )

    OR NOT (
            tgt.periodo_key
            <=>
            src.periodo_key
       )

    OR NOT (
            tgt.planta_key
            <=>
            src.planta_key
       )

    OR NOT (
            tgt.disponibilidad_real_kwh
            <=>
            src.disponibilidad_real_kwh
       )

    OR NOT (
            tgt.version_seleccionada
            <=>
            src.version_seleccionada
       )

    OR NOT (
            tgt.prioridad_version
            <=>
            src.prioridad_version
       )
)

THEN UPDATE SET
    tgt.fecha_key =
        src.fecha_key,

    tgt.periodo_key =
        src.periodo_key,

    tgt.planta_key =
        src.planta_key,

    tgt.fecha_hora =
        src.fecha_hora,

    tgt.disponibilidad_real_kwh =
        src.disponibilidad_real_kwh,

    tgt.version_seleccionada =
        src.version_seleccionada,

    tgt.prioridad_version =
        src.prioridad_version,

    tgt.fecha_actualizacion =
        src.fecha_proceso

WHEN NOT MATCHED THEN
INSERT (
    disponibilidad_key,
    fecha_key,
    periodo_key,
    planta_key,
    fecha_hora,
    disponibilidad_real_kwh,
    version_seleccionada,
    prioridad_version,
    fecha_creacion,
    fecha_actualizacion
)
VALUES (
    src.disponibilidad_key,
    src.fecha_key,
    src.periodo_key,
    src.planta_key,
    src.fecha_hora,
    src.disponibilidad_real_kwh,
    src.version_seleccionada,
    src.prioridad_version,
    src.fecha_proceso,
    src.fecha_proceso
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_fact_precio_bolsa AS

WITH precio_base AS (
    SELECT
        id,
        fecha_hora,

        UPPER(TRIM(codigo_variable))
            AS codigo_variable,

        UPPER(TRIM(version))
            AS version,

        CAST(valor AS DECIMAL(18,6))
            AS valor_cop_kwh,

        ingestion_timestamp,
        load_date,
        silver_updated_at,

        CASE
            /*
            TX3, TX4, TX5... son ajustes posteriores.
            Cuanto mayor sea TXn, mayor será su prioridad.
            */
            WHEN UPPER(TRIM(version)) RLIKE '^TX[0-9]+$'
             AND CAST(
                    REGEXP_EXTRACT(
                        UPPER(TRIM(version)),
                        '^TX([0-9]+)$',
                        1
                    ) AS INT
                 ) >= 3
            THEN
                1000
                + CAST(
                    REGEXP_EXTRACT(
                        UPPER(TRIM(version)),
                        '^TX([0-9]+)$',
                        1
                    ) AS INT
                )

            WHEN UPPER(TRIM(version)) = 'TXF'
                THEN 900

            WHEN UPPER(TRIM(version)) = 'TXR'
                THEN 800

            WHEN UPPER(TRIM(version)) = 'TX2'
                THEN 200

            WHEN UPPER(TRIM(version)) = 'TX1'
                THEN 100

            ELSE 0
        END AS prioridad_version

    FROM observatorio_dev.silver.precio_bolsa

    WHERE UPPER(TRIM(codigo_variable))
          IN ('PB_INT', 'PB_NAL', 'PB_TIE')

      AND UPPER(TRIM(codigo_duracion)) = 'PT1H'

      AND UPPER(TRIM(unidad_medida)) = 'COP/KWH'

      AND fecha_hora IS NOT NULL
      AND valor IS NOT NULL
),

/*
Selecciona una única versión para cada variable y hora.
*/
precio_versionado AS (
    SELECT
        *,

        ROW_NUMBER() OVER (
            PARTITION BY
                fecha_hora,
                codigo_variable

            ORDER BY
                prioridad_version DESC,
                silver_updated_at DESC NULLS LAST,
                ingestion_timestamp DESC NULLS LAST,
                load_date DESC NULLS LAST,
                id DESC
        ) AS posicion_version

    FROM precio_base
),

precio_seleccionado AS (
    SELECT
        fecha_hora,
        codigo_variable,
        valor_cop_kwh,
        version,
        prioridad_version

    FROM precio_versionado

    WHERE posicion_version = 1
),

/*
Convierte las tres variables en columnas.
*/
precio_pivot AS (
    SELECT
        fecha_hora,

        MAX(
            CASE
                WHEN codigo_variable = 'PB_INT'
                THEN valor_cop_kwh
            END
        ) AS precio_bolsa_internacional_cop_kwh,

        MAX(
            CASE
                WHEN codigo_variable = 'PB_NAL'
                THEN valor_cop_kwh
            END
        ) AS precio_bolsa_nacional_cop_kwh,

        MAX(
            CASE
                WHEN codigo_variable = 'PB_TIE'
                THEN valor_cop_kwh
            END
        ) AS precio_bolsa_tie_cop_kwh,

        MAX(
            CASE
                WHEN codigo_variable = 'PB_INT'
                THEN version
            END
        ) AS version_pb_int,

        MAX(
            CASE
                WHEN codigo_variable = 'PB_INT'
                THEN prioridad_version
            END
        ) AS prioridad_pb_int,

        MAX(
            CASE
                WHEN codigo_variable = 'PB_NAL'
                THEN version
            END
        ) AS version_pb_nal,

        MAX(
            CASE
                WHEN codigo_variable = 'PB_NAL'
                THEN prioridad_version
            END
        ) AS prioridad_pb_nal,

        MAX(
            CASE
                WHEN codigo_variable = 'PB_TIE'
                THEN version
            END
        ) AS version_pb_tie,

        MAX(
            CASE
                WHEN codigo_variable = 'PB_TIE'
                THEN prioridad_version
            END
        ) AS prioridad_pb_tie

    FROM precio_seleccionado

    GROUP BY fecha_hora
)

SELECT
    SHA2(
        CONCAT_WS(
            '||',
            DATE_FORMAT(
                fecha_hora,
                'yyyy-MM-dd HH:mm:ss'
            ),
            'PRECIO_BOLSA'
        ),
        256
    ) AS precio_bolsa_key,

    CAST(
        DATE_FORMAT(fecha_hora, 'yyyyMMdd')
        AS INT
    ) AS fecha_key,

    CAST(
        HOUR(fecha_hora) + 1
        AS TINYINT
    ) AS periodo_key,

    fecha_hora,

    precio_bolsa_internacional_cop_kwh,
    precio_bolsa_nacional_cop_kwh,
    precio_bolsa_tie_cop_kwh,

    version_pb_int,
    prioridad_pb_int,

    version_pb_nal,
    prioridad_pb_nal,

    version_pb_tie,
    prioridad_pb_tie

FROM precio_pivot;


In [0]:
%sql
SELECT
    COUNT(*) AS registros_staging,

    COUNT(DISTINCT precio_bolsa_key)
        AS claves_distintas,

    SUM(
        CASE
            WHEN precio_bolsa_internacional_cop_kwh IS NULL
            THEN 1
            ELSE 0
        END
    ) AS pb_int_faltantes,

    SUM(
        CASE
            WHEN precio_bolsa_nacional_cop_kwh IS NULL
            THEN 1
            ELSE 0
        END
    ) AS pb_nal_faltantes,

    SUM(
        CASE
            WHEN precio_bolsa_tie_cop_kwh IS NULL
            THEN 1
            ELSE 0
        END
    ) AS pb_tie_faltantes

FROM stg_fact_precio_bolsa;

In [0]:
%sql
SELECT
    SUM(
        CASE
            WHEN d.fecha_key IS NULL
            THEN 1
            ELSE 0
        END
    ) AS fechas_sin_dimension,

    SUM(
        CASE
            WHEN p.periodo_key IS NULL
            THEN 1
            ELSE 0
        END
    ) AS periodos_sin_dimension

FROM stg_fact_precio_bolsa AS s

LEFT JOIN observatorio_dev.gold.dim_fecha AS d
    ON s.fecha_key = d.fecha_key

LEFT JOIN observatorio_dev.gold.dim_periodo AS p
    ON s.periodo_key = p.periodo_key;

In [0]:
%sql
MERGE INTO observatorio_dev.gold.fact_precio_bolsa AS tgt

USING (
    SELECT
        precio_bolsa_key,
        fecha_key,
        periodo_key,
        fecha_hora,

        precio_bolsa_internacional_cop_kwh,
        precio_bolsa_nacional_cop_kwh,
        precio_bolsa_tie_cop_kwh,

        version_pb_int,
        prioridad_pb_int,

        version_pb_nal,
        prioridad_pb_nal,

        version_pb_tie,
        prioridad_pb_tie,

        CURRENT_TIMESTAMP() AS fecha_proceso

    FROM stg_fact_precio_bolsa
) AS src

ON tgt.precio_bolsa_key = src.precio_bolsa_key

WHEN MATCHED AND (
       NOT (tgt.fecha_key <=> src.fecha_key)
    OR NOT (tgt.periodo_key <=> src.periodo_key)

    OR NOT (
        tgt.precio_bolsa_internacional_cop_kwh
        <=>
        src.precio_bolsa_internacional_cop_kwh
    )

    OR NOT (
        tgt.precio_bolsa_nacional_cop_kwh
        <=>
        src.precio_bolsa_nacional_cop_kwh
    )

    OR NOT (
        tgt.precio_bolsa_tie_cop_kwh
        <=>
        src.precio_bolsa_tie_cop_kwh
    )

    OR NOT (tgt.version_pb_int <=> src.version_pb_int)
    OR NOT (tgt.prioridad_pb_int <=> src.prioridad_pb_int)

    OR NOT (tgt.version_pb_nal <=> src.version_pb_nal)
    OR NOT (tgt.prioridad_pb_nal <=> src.prioridad_pb_nal)

    OR NOT (tgt.version_pb_tie <=> src.version_pb_tie)
    OR NOT (tgt.prioridad_pb_tie <=> src.prioridad_pb_tie)
)

THEN UPDATE SET
    tgt.fecha_key =
        src.fecha_key,

    tgt.periodo_key =
        src.periodo_key,

    tgt.fecha_hora =
        src.fecha_hora,

    tgt.precio_bolsa_internacional_cop_kwh =
        src.precio_bolsa_internacional_cop_kwh,

    tgt.precio_bolsa_nacional_cop_kwh =
        src.precio_bolsa_nacional_cop_kwh,

    tgt.precio_bolsa_tie_cop_kwh =
        src.precio_bolsa_tie_cop_kwh,

    tgt.version_pb_int =
        src.version_pb_int,

    tgt.prioridad_pb_int =
        src.prioridad_pb_int,

    tgt.version_pb_nal =
        src.version_pb_nal,

    tgt.prioridad_pb_nal =
        src.prioridad_pb_nal,

    tgt.version_pb_tie =
        src.version_pb_tie,

    tgt.prioridad_pb_tie =
        src.prioridad_pb_tie,

    tgt.fecha_actualizacion =
        src.fecha_proceso

WHEN NOT MATCHED THEN
INSERT (
    precio_bolsa_key,
    fecha_key,
    periodo_key,
    fecha_hora,

    precio_bolsa_internacional_cop_kwh,
    precio_bolsa_nacional_cop_kwh,
    precio_bolsa_tie_cop_kwh,

    version_pb_int,
    prioridad_pb_int,

    version_pb_nal,
    prioridad_pb_nal,

    version_pb_tie,
    prioridad_pb_tie,

    fecha_creacion,
    fecha_actualizacion
)
VALUES (
    src.precio_bolsa_key,
    src.fecha_key,
    src.periodo_key,
    src.fecha_hora,

    src.precio_bolsa_internacional_cop_kwh,
    src.precio_bolsa_nacional_cop_kwh,
    src.precio_bolsa_tie_cop_kwh,

    src.version_pb_int,
    src.prioridad_pb_int,

    src.version_pb_nal,
    src.prioridad_pb_nal,

    src.version_pb_tie,
    src.prioridad_pb_tie,

    src.fecha_proceso,
    src.fecha_proceso
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW
stg_fact_energia_embalsada_planta AS

WITH nivel_base AS (
    SELECT
        id,

        CAST(fecha_inicio AS DATE)
            AS fecha_medicion,

        UPPER(TRIM(codigo_planta))
            AS codigo_planta,

        UPPER(TRIM(codigo_variable))
            AS codigo_variable,

        UPPER(TRIM(version))
            AS version,

        CAST(valor AS DECIMAL(20,4))
            AS energia_embalsada_kwh,

        ingestion_timestamp,
        load_date,
        silver_updated_at,

        CASE
            WHEN UPPER(TRIM(version)) RLIKE '^TX[0-9]+$'
             AND CAST(
                    REGEXP_EXTRACT(
                        UPPER(TRIM(version)),
                        '^TX([0-9]+)$',
                        1
                    ) AS INT
                 ) >= 3
            THEN
                1000
                + CAST(
                    REGEXP_EXTRACT(
                        UPPER(TRIM(version)),
                        '^TX([0-9]+)$',
                        1
                    ) AS INT
                )

            WHEN UPPER(TRIM(version)) = 'TXF'
                THEN 900

            WHEN UPPER(TRIM(version)) = 'TXR'
                THEN 800

            WHEN UPPER(TRIM(version)) = 'TX2'
                THEN 200

            WHEN UPPER(TRIM(version)) = 'TX1'
                THEN 100

            ELSE 0
        END AS prioridad_version

    FROM observatorio_dev.silver.niveles_embalses

    WHERE UPPER(TRIM(codigo_variable)) = 'NEM'
      AND UPPER(TRIM(codigo_duracion)) = 'P1D'
      AND UPPER(TRIM(unidad_medida)) = 'KWH'
      AND fecha_inicio IS NOT NULL
      AND codigo_planta IS NOT NULL
      AND TRIM(codigo_planta) <> ''
      AND valor IS NOT NULL
),

nivel_versionado AS (
    SELECT
        *,

        ROW_NUMBER() OVER (
            PARTITION BY
                fecha_medicion,
                codigo_planta,
                codigo_variable

            ORDER BY
                prioridad_version DESC,
                silver_updated_at DESC NULLS LAST,
                ingestion_timestamp DESC NULLS LAST,
                load_date DESC NULLS LAST,
                id DESC
        ) AS posicion_version

    FROM nivel_base
),

mediciones AS (
    SELECT
        SHA2(
            CONCAT_WS(
                '||',
                CAST(fecha_medicion AS STRING),
                codigo_planta,
                codigo_variable
            ),
            256
        ) AS energia_embalsada_key,

        CAST(
            DATE_FORMAT(
                fecha_medicion,
                'yyyyMMdd'
            ) AS INT
        ) AS fecha_key,

        fecha_medicion,
        codigo_planta,
        energia_embalsada_kwh,

        version
            AS version_seleccionada,

        prioridad_version

    FROM nivel_versionado

    WHERE posicion_version = 1
)

SELECT
    m.energia_embalsada_key,
    m.fecha_key,
    p.planta_key,
    m.fecha_medicion,
    m.energia_embalsada_kwh,
    m.version_seleccionada,
    m.prioridad_version

FROM mediciones AS m

LEFT JOIN observatorio_dev.gold.dim_planta AS p
    ON m.codigo_planta = p.codigo_planta

   AND m.fecha_medicion
       BETWEEN p.fecha_inicio
           AND p.fecha_fin;

In [0]:
%sql
SELECT
    COUNT(*) AS registros_staging,

    COUNT(DISTINCT energia_embalsada_key)
        AS claves_distintas,

    SUM(
        CASE
            WHEN planta_key IS NULL
            THEN 1
            ELSE 0
        END
    ) AS plantas_sin_dimension,

    SUM(
        CASE
            WHEN fecha_key IS NULL
            THEN 1
            ELSE 0
        END
    ) AS fechas_nulas,

    MIN(fecha_medicion) AS fecha_minima,
    MAX(fecha_medicion) AS fecha_maxima

FROM stg_fact_energia_embalsada_planta;

In [0]:
%sql
MERGE INTO
observatorio_dev.gold.fact_energia_embalsada_planta AS tgt

USING (
    SELECT
        energia_embalsada_key,
        fecha_key,
        planta_key,
        fecha_medicion,
        energia_embalsada_kwh,
        version_seleccionada,
        prioridad_version,
        CURRENT_TIMESTAMP() AS fecha_proceso

    FROM stg_fact_energia_embalsada_planta

    WHERE planta_key IS NOT NULL
) AS src

ON tgt.energia_embalsada_key =
   src.energia_embalsada_key

WHEN MATCHED AND (
       NOT (
            tgt.fecha_key
            <=>
            src.fecha_key
       )

    OR NOT (
            tgt.planta_key
            <=>
            src.planta_key
       )

    OR NOT (
            tgt.fecha_medicion
            <=>
            src.fecha_medicion
       )

    OR NOT (
            tgt.energia_embalsada_kwh
            <=>
            src.energia_embalsada_kwh
       )

    OR NOT (
            tgt.version_seleccionada
            <=>
            src.version_seleccionada
       )

    OR NOT (
            tgt.prioridad_version
            <=>
            src.prioridad_version
       )
)

THEN UPDATE SET
    tgt.fecha_key =
        src.fecha_key,

    tgt.planta_key =
        src.planta_key,

    tgt.fecha_medicion =
        src.fecha_medicion,

    tgt.energia_embalsada_kwh =
        src.energia_embalsada_kwh,

    tgt.version_seleccionada =
        src.version_seleccionada,

    tgt.prioridad_version =
        src.prioridad_version,

    tgt.fecha_actualizacion =
        src.fecha_proceso

WHEN NOT MATCHED THEN
INSERT (
    energia_embalsada_key,
    fecha_key,
    planta_key,
    fecha_medicion,
    energia_embalsada_kwh,
    version_seleccionada,
    prioridad_version,
    fecha_creacion,
    fecha_actualizacion
)
VALUES (
    src.energia_embalsada_key,
    src.fecha_key,
    src.planta_key,
    src.fecha_medicion,
    src.energia_embalsada_kwh,
    src.version_seleccionada,
    src.prioridad_version,
    src.fecha_proceso,
    src.fecha_proceso
);

In [0]:
%sql
SELECT
    'fact_generacion_real' AS tabla,
    COUNT(*) AS registros,
    COUNT(DISTINCT generacion_key) AS claves_distintas
FROM observatorio_dev.gold.fact_generacion_real

UNION ALL

SELECT
    'fact_disponibilidad_planta',
    COUNT(*),
    COUNT(DISTINCT disponibilidad_key)
FROM observatorio_dev.gold.fact_disponibilidad_planta

UNION ALL

SELECT
    'fact_precio_bolsa',
    COUNT(*),
    COUNT(DISTINCT precio_bolsa_key)
FROM observatorio_dev.gold.fact_precio_bolsa

UNION ALL

SELECT
    'fact_energia_embalsada_planta',
    COUNT(*),
    COUNT(DISTINCT energia_embalsada_key)
FROM observatorio_dev.gold.fact_energia_embalsada_planta;

In [0]:
%sql
SELECT
    'generacion' AS hecho,
    MIN(fecha_hora) AS fecha_minima,
    MAX(fecha_hora) AS fecha_maxima
FROM observatorio_dev.gold.fact_generacion_real

UNION ALL

SELECT
    'disponibilidad',
    MIN(fecha_hora),
    MAX(fecha_hora)
FROM observatorio_dev.gold.fact_disponibilidad_planta

UNION ALL

SELECT
    'precio_bolsa',
    MIN(fecha_hora),
    MAX(fecha_hora)
FROM observatorio_dev.gold.fact_precio_bolsa

UNION ALL

SELECT
    'energia_embalsada',
    CAST(MIN(fecha_medicion) AS TIMESTAMP),
    CAST(MAX(fecha_medicion) AS TIMESTAMP)
FROM observatorio_dev.gold.fact_energia_embalsada_planta;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW ref_alias_planta_embalse AS

SELECT *
FROM VALUES
    (
        'PLANTA',
        'SOGAMOSOS',
        'SOGAMOSO'
    ),

    (
        'EMBALSE',
        'CALIMA1',
        'CALIMA'
    ),

    (
        'EMBALSE',
        'URRA1',
        'URRA'
    ),

    (
        'EMBALSE',
        'TOPOROCO',
        'TOPOCORO'
    ),

    (
        'EMBALSE',
        'PORCEII',
        'PORCE2'
    ),

    (
        'EMBALSE',
        'PORCEIII',
        'PORCE3'
    )

AS alias (
    tipo_entidad,
    nombre_fuente_normalizado,
    nombre_destino_normalizado
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_validacion_planta_embalse AS

WITH relaciones_fuente AS (
    SELECT DISTINCT
        UPPER(TRIM(region)) AS region,

        TRIM(nombre_planta)
            AS nombre_planta_fuente,

        TRIM(nombre_reservorio)
            AS nombre_reservorio_fuente,

        TRANSLATE(
            REGEXP_REPLACE(
                UPPER(TRIM(nombre_planta)),
                '[^A-Z0-9ÁÉÍÓÚÜÑ]',
                ''
            ),
            'ÁÉÍÓÚÜÑ',
            'AEIOUUN'
        ) AS nombre_planta_original,

        TRANSLATE(
            REGEXP_REPLACE(
                UPPER(TRIM(nombre_reservorio)),
                '[^A-Z0-9ÁÉÍÓÚÜÑ]',
                ''
            ),
            'ÁÉÍÓÚÜÑ',
            'AEIOUUN'
        ) AS nombre_embalse_original

    FROM observatorio_dev.silver.plantas_reservorios

    WHERE nombre_planta IS NOT NULL
      AND TRIM(nombre_planta) <> ''
      AND nombre_reservorio IS NOT NULL
      AND TRIM(nombre_reservorio) <> ''
),

relaciones_con_alias AS (
    SELECT
        f.region,
        f.nombre_planta_fuente,
        f.nombre_reservorio_fuente,

        COALESCE(
            ap.nombre_destino_normalizado,
            f.nombre_planta_original
        ) AS nombre_planta_normalizado,

        COALESCE(
            ae.nombre_destino_normalizado,
            f.nombre_embalse_original
        ) AS nombre_embalse_normalizado

    FROM relaciones_fuente AS f

    LEFT JOIN ref_alias_planta_embalse AS ap
        ON ap.tipo_entidad = 'PLANTA'
       AND f.nombre_planta_original =
           ap.nombre_fuente_normalizado

    LEFT JOIN ref_alias_planta_embalse AS ae
        ON ae.tipo_entidad = 'EMBALSE'
       AND f.nombre_embalse_original =
           ae.nombre_fuente_normalizado
),

plantas AS (
    SELECT
        planta_key,
        codigo_planta,
        nombre_planta,

        TRANSLATE(
            REGEXP_REPLACE(
                UPPER(TRIM(nombre_planta)),
                '[^A-Z0-9ÁÉÍÓÚÜÑ]',
                ''
            ),
            'ÁÉÍÓÚÜÑ',
            'AEIOUUN'
        ) AS nombre_planta_normalizado

    FROM observatorio_dev.gold.dim_planta

    WHERE es_actual = TRUE
),

embalses AS (
    SELECT
        embalse_key,
        codigo_embalse,
        nombre_embalse,

        TRANSLATE(
            REGEXP_REPLACE(
                UPPER(TRIM(nombre_embalse)),
                '[^A-Z0-9ÁÉÍÓÚÜÑ]',
                ''
            ),
            'ÁÉÍÓÚÜÑ',
            'AEIOUUN'
        ) AS nombre_embalse_normalizado

    FROM observatorio_dev.gold.dim_embalse
),

cobertura AS (
    SELECT
        f.region,
        f.nombre_planta_fuente,
        f.nombre_reservorio_fuente,
        f.nombre_planta_normalizado,
        f.nombre_embalse_normalizado,

        COUNT(DISTINCT p.codigo_planta)
            AS plantas_encontradas,

        COUNT(DISTINCT e.embalse_key)
            AS embalses_encontrados,

        SORT_ARRAY(
            COLLECT_SET(p.codigo_planta)
        ) AS codigos_planta,

        SORT_ARRAY(
            COLLECT_SET(e.codigo_embalse)
        ) AS codigos_embalse,

        SORT_ARRAY(
            COLLECT_SET(p.nombre_planta)
        ) AS nombres_planta_dimension,

        SORT_ARRAY(
            COLLECT_SET(e.nombre_embalse)
        ) AS nombres_embalse_dimension

    FROM relaciones_con_alias AS f

    LEFT JOIN plantas AS p
        ON f.nombre_planta_normalizado =
           p.nombre_planta_normalizado

    LEFT JOIN embalses AS e
        ON f.nombre_embalse_normalizado =
           e.nombre_embalse_normalizado

    GROUP BY
        f.region,
        f.nombre_planta_fuente,
        f.nombre_reservorio_fuente,
        f.nombre_planta_normalizado,
        f.nombre_embalse_normalizado
)

SELECT
    *,

    CASE
        WHEN plantas_encontradas = 1
         AND embalses_encontrados = 1
            THEN 'MATCH_UNICO'

        WHEN plantas_encontradas = 0
         AND embalses_encontrados = 0
            THEN 'PLANTA_Y_EMBALSE_NO_ENCONTRADOS'

        WHEN plantas_encontradas = 0
            THEN 'PLANTA_NO_ENCONTRADA'

        WHEN embalses_encontrados = 0
            THEN 'EMBALSE_NO_ENCONTRADO'

        WHEN plantas_encontradas > 1
            THEN 'PLANTA_AMBIGUA'

        WHEN embalses_encontrados > 1
            THEN 'EMBALSE_AMBIGUO'

        ELSE 'REVISAR'
    END AS estado_match

FROM cobertura;

In [0]:
%sql
SELECT
    estado_match,
    COUNT(*) AS relaciones

FROM stg_validacion_planta_embalse

GROUP BY estado_match

ORDER BY estado_match;

# Bridge tables para juntar dim embalse y plantas

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_bridge_planta_embalse AS

WITH relaciones_resueltas AS (
    SELECT
        region,
        nombre_planta_fuente,
        nombre_reservorio_fuente,

        ELEMENT_AT(
            codigos_planta,
            1
        ) AS codigo_planta,

        ELEMENT_AT(
            codigos_embalse,
            1
        ) AS codigo_embalse

    FROM stg_validacion_planta_embalse

    WHERE estado_match = 'MATCH_UNICO'
),

cardinalidad AS (
    SELECT
        codigo_planta,

        COUNT(
            DISTINCT codigo_embalse
        ) AS cantidad_embalses_planta

    FROM relaciones_resueltas

    GROUP BY codigo_planta
),

relaciones_con_cardinalidad AS (
    SELECT
        r.region,
        r.nombre_planta_fuente,
        r.nombre_reservorio_fuente,
        r.codigo_planta,
        r.codigo_embalse,
        c.cantidad_embalses_planta

    FROM relaciones_resueltas AS r

    INNER JOIN cardinalidad AS c
        ON r.codigo_planta = c.codigo_planta
)

SELECT
    SHA2(
        CONCAT_WS(
            '||',
            CAST(p.planta_key AS STRING),
            CAST(e.embalse_key AS STRING)
        ),
        256
    ) AS planta_embalse_key,

    p.planta_key,
    e.embalse_key,

    p.codigo_planta,
    e.codigo_embalse,

    r.region,
    r.nombre_planta_fuente,
    r.nombre_reservorio_fuente,

    r.cantidad_embalses_planta,

    r.cantidad_embalses_planta = 1
        AS es_relacion_unica

FROM relaciones_con_cardinalidad AS r

INNER JOIN observatorio_dev.gold.dim_planta AS p
    ON r.codigo_planta = p.codigo_planta

INNER JOIN observatorio_dev.gold.dim_embalse AS e
    ON r.codigo_embalse = e.codigo_embalse;

In [0]:
%sql
SELECT
    COUNT(*) AS relaciones_totales,

    COUNT(DISTINCT planta_embalse_key)
        AS claves_distintas,

    COUNT(DISTINCT codigo_planta)
        AS plantas,

    COUNT(DISTINCT codigo_embalse)
        AS embalses

FROM stg_bridge_planta_embalse;

In [0]:
%sql
SELECT
    planta_key,
    embalse_key,
    COUNT(*) AS cantidad

FROM stg_bridge_planta_embalse

GROUP BY
    planta_key,
    embalse_key

HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT
    codigo_planta,
    nombre_planta_fuente,
    cantidad_embalses_planta,

    SORT_ARRAY(
        COLLECT_SET(codigo_embalse)
    ) AS codigos_embalse,

    SORT_ARRAY(
        COLLECT_SET(nombre_reservorio_fuente)
    ) AS reservorios

FROM stg_bridge_planta_embalse

WHERE cantidad_embalses_planta > 1

GROUP BY
    codigo_planta,
    nombre_planta_fuente,
    cantidad_embalses_planta

ORDER BY
    cantidad_embalses_planta DESC,
    codigo_planta;

In [0]:
%sql
MERGE INTO
observatorio_dev.gold.bridge_planta_embalse AS tgt

USING (
    SELECT
        planta_embalse_key,
        planta_key,
        embalse_key,
        codigo_planta,
        codigo_embalse,
        region,
        nombre_planta_fuente,
        nombre_reservorio_fuente,
        cantidad_embalses_planta,
        es_relacion_unica,
        CURRENT_TIMESTAMP() AS fecha_proceso

    FROM stg_bridge_planta_embalse
) AS src

ON tgt.planta_embalse_key = src.planta_embalse_key

WHEN MATCHED AND (
       NOT (tgt.planta_key <=> src.planta_key)
    OR NOT (tgt.embalse_key <=> src.embalse_key)
    OR NOT (tgt.codigo_planta <=> src.codigo_planta)
    OR NOT (tgt.codigo_embalse <=> src.codigo_embalse)
    OR NOT (tgt.region <=> src.region)
    OR NOT (
        tgt.nombre_planta_fuente
        <=>
        src.nombre_planta_fuente
    )
    OR NOT (
        tgt.nombre_reservorio_fuente
        <=>
        src.nombre_reservorio_fuente
    )
    OR NOT (
        tgt.cantidad_embalses_planta
        <=>
        src.cantidad_embalses_planta
    )
    OR NOT (
        tgt.es_relacion_unica
        <=>
        src.es_relacion_unica
    )
)

THEN UPDATE SET
    tgt.planta_key =
        src.planta_key,

    tgt.embalse_key =
        src.embalse_key,

    tgt.codigo_planta =
        src.codigo_planta,

    tgt.codigo_embalse =
        src.codigo_embalse,

    tgt.region =
        src.region,

    tgt.nombre_planta_fuente =
        src.nombre_planta_fuente,

    tgt.nombre_reservorio_fuente =
        src.nombre_reservorio_fuente,

    tgt.cantidad_embalses_planta =
        src.cantidad_embalses_planta,

    tgt.es_relacion_unica =
        src.es_relacion_unica,

    tgt.fecha_actualizacion =
        src.fecha_proceso

WHEN NOT MATCHED THEN
INSERT (
    planta_embalse_key,
    planta_key,
    embalse_key,
    codigo_planta,
    codigo_embalse,
    region,
    nombre_planta_fuente,
    nombre_reservorio_fuente,
    cantidad_embalses_planta,
    es_relacion_unica,
    fecha_creacion,
    fecha_actualizacion
)
VALUES (
    src.planta_embalse_key,
    src.planta_key,
    src.embalse_key,
    src.codigo_planta,
    src.codigo_embalse,
    src.region,
    src.nombre_planta_fuente,
    src.nombre_reservorio_fuente,
    src.cantidad_embalses_planta,
    src.es_relacion_unica,
    src.fecha_proceso,
    src.fecha_proceso
);

In [0]:
%sql
SET TIME ZONE 'UTC';

CREATE OR REPLACE TEMP VIEW stg_fact_demanda_real AS

WITH demanda_base AS (
    SELECT
        id,
        fecha_hora,
        codigo_agente,
        tipo_mercado,
        codigo_variable,
        codigo_duracion,
        unidad_medida,
        version,
        CAST(demanda_real_kwh AS DECIMAL(24,6))
            AS demanda_real_kwh,
        ingestion_timestamp,
        silver_updated_at,

        CASE
            WHEN version RLIKE '^TX[0-9]+$'
                 AND CAST(
                     REGEXP_EXTRACT(
                         version,
                         '^TX([0-9]+)$',
                         1
                     ) AS INT
                 ) >= 3
            THEN 1000 + CAST(
                REGEXP_EXTRACT(
                    version,
                    '^TX([0-9]+)$',
                    1
                ) AS INT
            )

            WHEN version = 'TXF' THEN 900
            WHEN version = 'TXR' THEN 800
            WHEN version = 'TX2' THEN 200
            WHEN version = 'TX1' THEN 100
            ELSE 0
        END AS prioridad_version

    FROM observatorio_dev.silver.demanda_real

    WHERE codigo_variable = 'DDAREAL'
      AND codigo_duracion = 'PT1H'
      AND unidad_medida = 'KWH'
      AND demanda_real_kwh >= 0
),

demanda_versionada AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY
                fecha_hora,
                codigo_agente,
                tipo_mercado,
                codigo_variable
            ORDER BY
                prioridad_version DESC,
                silver_updated_at DESC,
                ingestion_timestamp DESC,
                id DESC
        ) AS numero_version
    FROM demanda_base
),

demanda_seleccionada AS (
    SELECT *
    FROM demanda_versionada
    WHERE numero_version = 1
)

SELECT
    SHA2(
        CONCAT_WS(
            '||',
            DATE_FORMAT(
                demanda.fecha_hora,
                'yyyy-MM-dd HH:mm:ss'
            ),
            demanda.codigo_agente,
            demanda.tipo_mercado,
            demanda.codigo_variable
        ),
        256
    ) AS demanda_key,

    CAST(
        DATE_FORMAT(
            demanda.fecha_hora,
            'yyyyMMdd'
        ) AS INT
    ) AS fecha_key,

    CAST(
        HOUR(demanda.fecha_hora) + 1
        AS TINYINT
    ) AS periodo_key,

    agente.agente_key,

    demanda.fecha_hora,
    demanda.tipo_mercado,
    demanda.demanda_real_kwh,

    demanda.version AS version_seleccionada,
    demanda.prioridad_version,

    CURRENT_TIMESTAMP() AS fecha_creacion,
    CURRENT_TIMESTAMP() AS fecha_actualizacion

FROM demanda_seleccionada demanda

LEFT JOIN observatorio_dev.gold.dim_agente agente
    ON demanda.codigo_agente = agente.codigo_agente
   AND CAST(demanda.fecha_hora AS DATE)
       BETWEEN agente.fecha_inicio AND agente.fecha_fin;

In [0]:
%sql
SELECT
    COUNT(*) AS registros,
    COUNT(DISTINCT demanda_key) AS claves_distintas,
    COUNT(*) - COUNT(DISTINCT demanda_key) AS duplicados,
    SUM(
        CASE WHEN agente_key IS NULL THEN 1 ELSE 0 END
    ) AS agentes_sin_dimension,
    MIN(fecha_hora) AS fecha_minima,
    MAX(fecha_hora) AS fecha_maxima,
    COUNT(DISTINCT version_seleccionada)
        AS versiones_seleccionadas
FROM stg_fact_demanda_real;

In [0]:
%sql
SELECT
    SUM(
        CASE WHEN fecha.fecha_key IS NULL THEN 1 ELSE 0 END
    ) AS fechas_sin_dimension,

    SUM(
        CASE WHEN periodo.periodo_key IS NULL THEN 1 ELSE 0 END
    ) AS periodos_sin_dimension

FROM stg_fact_demanda_real demanda

LEFT JOIN observatorio_dev.gold.dim_fecha fecha
    ON demanda.fecha_key = fecha.fecha_key

LEFT JOIN observatorio_dev.gold.dim_periodo periodo
    ON demanda.periodo_key = periodo.periodo_key;

In [0]:
%sql
MERGE INTO observatorio_dev.gold.fact_demanda_real AS target

USING stg_fact_demanda_real AS source

ON target.demanda_key = source.demanda_key

WHEN MATCHED
AND NOT (
       target.fecha_key            <=> source.fecha_key
   AND target.periodo_key          <=> source.periodo_key
   AND target.agente_key           <=> source.agente_key
   AND target.fecha_hora           <=> source.fecha_hora
   AND target.tipo_mercado         <=> source.tipo_mercado
   AND target.demanda_real_kwh     <=> source.demanda_real_kwh
   AND target.version_seleccionada <=> source.version_seleccionada
   AND target.prioridad_version    <=> source.prioridad_version
)
THEN UPDATE SET
    target.fecha_key = source.fecha_key,
    target.periodo_key = source.periodo_key,
    target.agente_key = source.agente_key,
    target.fecha_hora = source.fecha_hora,
    target.tipo_mercado = source.tipo_mercado,
    target.demanda_real_kwh = source.demanda_real_kwh,
    target.version_seleccionada = source.version_seleccionada,
    target.prioridad_version = source.prioridad_version,
    target.fecha_actualizacion = CURRENT_TIMESTAMP()

WHEN NOT MATCHED
THEN INSERT (
    demanda_key,
    fecha_key,
    periodo_key,
    agente_key,
    fecha_hora,
    tipo_mercado,
    demanda_real_kwh,
    version_seleccionada,
    prioridad_version,
    fecha_creacion,
    fecha_actualizacion
)
VALUES (
    source.demanda_key,
    source.fecha_key,
    source.periodo_key,
    source.agente_key,
    source.fecha_hora,
    source.tipo_mercado,
    source.demanda_real_kwh,
    source.version_seleccionada,
    source.prioridad_version,
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP()
);